## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

### PART A: Divide our documents into chunks

In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [2]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


OpenAI API Key exists and begins sk-proj-


In [11]:
# How many characters in all the documents?

knowledge_base_path = "DALL_FULL_MD_Dataset/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(files)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")
#print(entire_knowledge_base)

['DALL_FULL_MD_Dataset\\company\\contact.md', 'DALL_FULL_MD_Dataset\\contracts\\contract_001_enterprise.md', 'DALL_FULL_MD_Dataset\\contracts\\contract_002_government.md', 'DALL_FULL_MD_Dataset\\contracts\\contract_003_datacenter.md', 'DALL_FULL_MD_Dataset\\contracts\\contract_004_maintenance.md', 'DALL_FULL_MD_Dataset\\contracts\\contract_005_cloud.md', 'DALL_FULL_MD_Dataset\\employees\\emp_001 john doe.md', 'DALL_FULL_MD_Dataset\\employees\\emp_002 priya sharma.md', 'DALL_FULL_MD_Dataset\\employees\\emp_003 rahul verma.md', 'DALL_FULL_MD_Dataset\\employees\\emp_004 sara khan.md', 'DALL_FULL_MD_Dataset\\employees\\emp_005 amit patnaik.md', 'DALL_FULL_MD_Dataset\\employees\\emp_006 neha singh.md', 'DALL_FULL_MD_Dataset\\employees\\emp_007 rohit mehta.md', 'DALL_FULL_MD_Dataset\\employees\\emp_008 ananya roy.md', 'DALL_FULL_MD_Dataset\\employees\\emp_009 kunal jain.md', 'DALL_FULL_MD_Dataset\\employees\\emp_010 meera nair.md', 'DALL_FULL_MD_Dataset\\products\\ai server.md', 'DALL_FULL_M

In [4]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 701


In [14]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("DALL_FULL_MD_Dataset/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    print(f"Loading documents from {doc_type} folder")
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    print(folder_docs)
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loading documents from company folder
[Document(metadata={'source': 'DALL_FULL_MD_Dataset\\company\\contact.md'}, page_content='DALL Technologies Pvt. Ltd.\nPhone: +91-80-4567-8900\nEmail: contact@dall.com\nSupport: support@dall.com\n')]
Loading documents from contracts folder
[Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_001_enterprise.md'}, page_content='Enterprise Supply Contract\nDuration: 3 Years\n'), Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_002_government.md'}, page_content='Government Infrastructure Contract\nDuration: 5 Years\n'), Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_003_datacenter.md'}, page_content='Data Center Partnership Contract\n'), Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_004_maintenance.md'}, page_content='Annual Maintenance Contract\n'), Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_005_cloud.md'}, page_content='Cloud Pro

In [6]:
documents[1]

Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_001_enterprise.md', 'doc_type': 'contracts'}, page_content='Enterprise Supply Contract\nDuration: 3 Years\n')

In [15]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(chunks)
print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

[Document(metadata={'source': 'DALL_FULL_MD_Dataset\\company\\contact.md', 'doc_type': 'company'}, page_content='DALL Technologies Pvt. Ltd.\nPhone: +91-80-4567-8900\nEmail: contact@dall.com\nSupport: support@dall.com'), Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_001_enterprise.md', 'doc_type': 'contracts'}, page_content='Enterprise Supply Contract\nDuration: 3 Years'), Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_002_government.md', 'doc_type': 'contracts'}, page_content='Government Infrastructure Contract\nDuration: 5 Years'), Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_003_datacenter.md', 'doc_type': 'contracts'}, page_content='Data Center Partnership Contract'), Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_004_maintenance.md', 'doc_type': 'contracts'}, page_content='Annual Maintenance Contract'), Document(metadata={'source': 'DALL_FULL_MD_Dataset\\contracts\\contract_005_c

In [8]:
chunks[15]

Document(metadata={'source': 'DALL_FULL_MD_Dataset\\employees\\emp_010 meera nair.md', 'doc_type': 'employees'}, page_content='# Employee Profile: Meera Nair\nRole: Documentation Lead\nSkills: Markdown, Docs-as-code, Technical writing\nCareer: Joined in 2019, owns documentation standards')

### PART B: Make vectors and store in Chroma

In Week 3, you set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).

In [9]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 26 documents


In [10]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 26 vectors with 384 dimensions in the vector store


### Part C: Visualize!

In [ ]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)
n_samples = vectors.shape[0]
tsne = TSNE(n_components=2,perplexity=min(10, n_samples - 1), random_state=12)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# Let's try 3D!

n_samples = vectors.shape[0]
tsne = TSNE(n_components=3,perplexity=min(10, n_samples - 1), random_state=12)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()